In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json
!pip install openai
from openai import OpenAI

In [ ]:
cmd = pd.read_csv("CLIENT 1 combined_data(2025-3-1-2026-2-28).csv")
chwpd  = pd.read_csv("channel-wise-publishing duration.csv")
chwp = pd.read_csv("channel-wise-publishing.csv")
cmdbchu = pd.read_csv("combined_data(2025-3-1-2026-2-28) by channel and user.csv")
cmdbit = pd.read_csv("combined_data(2025-3-1-2026-2-28) by input type.csv")
cmdbl = pd.read_csv("combined_data(2025-3-1-2026-2-28) by language.csv")
cmdbot = pd.read_csv("combined_data(2025-3-1-2026-2-28) by output type.csv")
cmdbu = pd.read_csv("combined_data(2025-3-1-2026-2-28) by user.csv")
mc = pd.read_csv("monthly-chart.csv")

In [ ]:
disc = {
    "cmd": "Channel-wise aggregated metrics for 18 channels (labeled A-R). Contains Uploaded Count, Created Count, Published Count, and their respective durations in seconds. Use this for comparing overall performance across channels or getting totals per channel.",
    "chwp": "Number of video clips published by each of the 18 channels (labeled A-R) broken down by social media platform — Facebook, Instagram, LinkedIn, Reels, Shorts, X, YouTube, Threads. Use this for platform-wise publishing counts per channel.",
    "chwpd": "Total published duration (in seconds) for each of the 18 channels (labeled A-R) broken down by social media platform — Facebook Duration, Instagram Duration, LinkedIn Duration, Reels Duration, Shorts Duration, X Duration, YouTube Duration, Threads Duration. Use this for platform-wise publishing duration per channel.",
    "cmdbchu": "Granular metrics for each Channel-User combination (236 rows). Contains Uploaded Count, Created Count, Published Count and their durations in seconds. Use this when query involves both a specific channel and a specific user, or to compare users within a channel.",
    "cmdbit": "Aggregated metrics grouped by input type of the source video. Input types include: interview, news bulletin, special reports, speech, debate, press conference, discussion show, sports show, unknown, podcast, drama, in-brief. Contains counts and durations. Use this for any query about input/source content types.",
    "cmdbl": "Aggregated metrics grouped by language of the content. Languages include: en (English), hi (Hindi), mix, es (Spanish), ar (Arabic), mr (Marathi). Contains counts and durations. Use this for any language-wise analysis.",
    "cmdbot": "Aggregated metrics grouped by output type of the processed video. Output types include: Full package, Key moments, Summary, Chapters, My key moments. Contains counts and durations. Use this for any query about output/clip types.",
    "cmdbu": "Aggregated metrics for each individual user (45 users). Contains Uploaded Count, Created Count, Published Count and their durations in seconds. Use this for user-wise performance comparison or to find top/bottom performing users.",
    "mc": "Month-wise aggregated count metrics across 12 months (Apr 2025 to Mar 2026). Contains Total Uploaded count, Total Created count, Total Published count, Total Uploaded Duration, Total Created Duration, Total Published Duration in seconds. Use this for monthly trends in video counts and duration."
}

In [ ]:
datasets = {
    "cmd": cmd, "chwpd": chwpd, "chwp": chwp, "cmdbchu": cmdbchu, "cmdbit": cmdbit, "cmdbl": cmdbl, "cmdbot": cmdbot, "cmdbu": cmdbu, "mc": mc
}

In [ ]:
kpi_framework = {
    "POAS": {
        "name": "Personal Output Amplification Score",
        "dimension": "AI Leverage",
        "formula": "Created Count / Uploaded Count (per user)",
        "benchmark": "≥ 3.5× per user",
        "current_avg": "2.98× (team avg across 42 active users)",
        "status": "MONITOR",
        "priority": "MONITOR",
        "datasets_required": ["cmdbu"],
        "description": "Measures how many AI-generated outputs each user's uploaded content produces on average. Higher score means the user uploads content that the AI can richly process.",
        "interpretation": "Score > 3.5× is good. Score < 2.0× means user is uploading content that AI cannot extract well from.",
        "action": "Train users with POAS < 2.0× on optimal upload formats. Interview and discussion content drives highest POAS."
    },

    "CPCR": {
        "name": "User Publish Conversion Rate",
        "dimension": "Publishing Outcome",
        "formula": "(Published Count / Created Count) × 100 (per user)",
        "benchmark": "≥ 15% per user",
        "current_avg": "0.74% team avg",
        "status": "CRITICAL",
        "priority": "IMMEDIATE",
        "datasets_required": ["cmdbu"],
        "description": "The percentage of each user's AI-created outputs that are actually published to an external platform. Most important individual performance metric.",
        "interpretation": "Only 13 of 45 users have CPCR > 0%. 32 users have never published a single piece.",
        "action": "Set minimum CPCR of 1% per active user as team SLA baseline. QA-Amit (13.64%) is the benchmark to target."
    },

    "CDI": {
        "name": "Content Depth Index",
        "dimension": "Content Quality",
        "formula": "Total Uploaded Duration (mins) / Uploaded Count (per user)",
        "benchmark": "≥ 15 min/upload",
        "current_avg": "10.6 min/upload team avg",
        "status": "REVIEW",
        "priority": "MEDIUM",
        "datasets_required": ["cmdbu"],
        "description": "Average duration in minutes of raw video each user uploads. Proxy for content quality and AI extraction potential — longer content yields richer AI derivatives.",
        "interpretation": "CDI > 15 min is ideal. Users with CDI < 5 min are uploading shallow content with limited AI extraction value.",
        "action": "Train low-CDI users to source longer interviews and debates. Users with CDI > 15 min need a dedicated publishing pipeline."
    },

    "WTE": {
        "name": "Workflow Throughput Efficiency",
        "dimension": "Operational Speed",
        "formula": "Total Created Count / Total Uploaded Duration (hours) (per user)",
        "benchmark": "≥ 35 outputs/hr",
        "current_avg": "~27 outputs/hr team avg",
        "status": "GOOD",
        "priority": "OPTIMISE",
        "datasets_required": ["cmdbu"],
        "description": "How many AI-generated outputs a user produces per hour of raw video uploaded. Duration-normalised productivity metric — fairest comparison of user-level efficiency.",
        "interpretation": "Harish leads at 41.3/hr. Users with WTE < 15/hr uploading short content may have AI feature adoption gaps.",
        "action": "Document Harish's workflow as team gold standard. Track WTE monthly to identify improving or declining users."
    },

    "CVS": {
        "name": "Channel Versatility Score",
        "dimension": "Cross-Channel Reach",
        "formula": "Count of distinct channels user has uploaded or created content for",
        "benchmark": "≥ 6 channels",
        "current_avg": "Avg 4.7 channels per active user",
        "status": "WARNING",
        "priority": "HIGH",
        "datasets_required": ["cmdbchu"],
        "description": "Counts how many distinct channels each user has worked on. Higher score means versatile operator capable of serving multiple editorial teams.",
        "interpretation": "Subhash leads with 17 channels but 0 published. vikas.s has best versatility-publish ratio at 40% CCPR.",
        "action": "Single-channel users should be encouraged to cross-train on at least 3 additional channels."
    },

    "ETR": {
        "name": "Publishing Punch Rate",
        "dimension": "Performance vs Peer Group",
        "formula": "ETR = (Published Count / Uploaded Count) × 100. Punch Rate = User ETR - Team avg ETR (2.49%)",
        "benchmark": "≥ 2× team average ETR (≥ 10%)",
        "current_avg": "2.49% team avg ETR",
        "status": "CRITICAL",
        "priority": "IMMEDIATE",
        "datasets_required": ["cmdbu"],
        "description": "Measures each user's upload-to-publish efficiency compared to team average. Positive punch rate means user publishes more per upload than average.",
        "interpretation": "Only 12 users have positive Punch Rate. QA-Amit (+57.5) and Harish (+14.6) are strongest performers.",
        "action": "Users with positive punch rate should mentor those at or below team average. Set 0 Punch Rate as minimum baseline expectation."
    },

    "CTDI": {
        "name": "Content Type Diversity Index",
        "dimension": "Creative Breadth",
        "formula": "Count of distinct input content types processed per user",
        "benchmark": "≥ 8 types out of 12",
        "current_avg": "Avg 5.2 types per active user",
        "status": "WARNING",
        "priority": "MEDIUM",
        "datasets_required": ["cmdbit", "cmdbchu"],
        "description": "Counts how many distinct input content types each user processes. Discussion-show (3.8% CPCR) and news bulletin (1.2% CPCR) have highest publish rates.",
        "interpretation": "Users stuck on speech (0.5% CPCR) or press conference (0.2% CPCR) face structural publish ceiling.",
        "action": "Route discussion-show and news bulletin content to users with strong publish pipelines. Sports Show, Podcast, Drama have 0 published — need dedicated pipeline."
    },

    "DPOR": {
        "name": "Dead-Pool Ownership Ratio",
        "dimension": "Waste Accountability",
        "formula": "(Created Count - Published Count) / Created Count × 100 (per user)",
        "benchmark": "< 80% per user",
        "current_avg": "99.26% team idle rate",
        "status": "CRITICAL",
        "priority": "IMMEDIATE",
        "datasets_required": ["cmdbu"],
        "description": "Percentage of each user's AI-created outputs that are never published — their personal content graveyard. Assigns individual accountability for idle inventory.",
        "interpretation": "QA-Amit (86.36%) is the only user below 90%. 26 users at exactly 100% DPOR — every output they created is unpublished.",
        "action": "Show each user their personal idle count on dashboard. Set 90-day dead-pool reduction target: reduce DPOR by 5 percentage points."
    },

    "DVY": {
        "name": "Duration Value Yield",
        "dimension": "Time ROI",
        "formula": "(Published Duration / Created Duration) × 100 (per user)",
        "benchmark": "≥ 10% utilised",
        "current_avg": "0.37% team utilisation",
        "status": "CRITICAL",
        "priority": "IMMEDIATE",
        "datasets_required": ["cmdbu"],
        "description": "Percentage of each user's total created content duration that has been published. Duration-space equivalent of CPCR — measures hours of content reaching audiences.",
        "interpretation": "QA-Amit leads at 5.80% DVY. 26 users have 0.00% DVY — every hour of AI-created content they hold is 100% idle.",
        "action": "Visualise idle hours per user on dashboard. Priority: users with high created duration but near-zero DVY have highest potential for quick wins."
    },

    "OUL": {
        "name": "Output-to-Upload Leverage Ratio",
        "dimension": "AI Utilisation Efficiency",
        "formula": "Total Created Count / Total Uploaded Count (per user)",
        "benchmark": "≥ 4.0× per user",
        "current_avg": "3.35× team avg",
        "status": "WARNING",
        "priority": "MEDIUM",
        "datasets_required": ["cmdbu"],
        "description": "How many AI outputs are created per raw video upload for each user. Identifies users who are maximally or minimally leveraging the AI platform.",
        "interpretation": "Deepika Uniyal leads at 5.78×. Ashween (1.04×) and Ritwik (1.04×) are essentially bypassing all AI features.",
        "action": "Users below 2.0× OUL should receive AI feature training. Help Ashween and Ritwik activate AI extraction features."
    },

    "PDE": {
        "name": "Publishing Duration Efficiency",
        "dimension": "Content Quality at Distribution",
        "formula": "Total Published Duration (mins) / Published Count (per user)",
        "benchmark": "≥ 3 min avg published video length",
        "current_avg": "1.9 min avg published video length",
        "status": "WARNING",
        "priority": "HIGH",
        "datasets_required": ["cmdbu"],
        "description": "Average duration in minutes of each user's published videos. Reveals the length profile of content each user publishes — short clips vs mid-length vs longer features.",
        "interpretation": "Adarsh (5.17 min) publishes longest content. Harish (0.75 min) publishes shortest — likely Shorts/Reels focused.",
        "action": "Match user PDE profiles to platform targets: Harish → Shorts, Adarsh → YouTube. Users with PDE < 1 min should test mid-length clips for LinkedIn."
    },

    "CCPR": {
        "name": "Cross-Channel Publish Rate",
        "dimension": "Channel Ownership",
        "formula": "(Channels with ≥1 Published / Channels Worked) × 100 (per user)",
        "benchmark": "≥ 50% of worked channels",
        "current_avg": "21% avg across active users",
        "status": "CRITICAL",
        "priority": "IMMEDIATE",
        "datasets_required": ["cmdbchu"],
        "description": "Of all channels a user has worked on, what percentage have they successfully published to at least once. Reveals whether multi-channel activity leads to multi-channel distribution.",
        "interpretation": "vikas.s leads at 40%. Subhash worked 17 channels but published on 0 — 0% CCPR across widest reach.",
        "action": "Set minimum CCPR target: every user should publish on ≥50% of assigned channels. Monthly CCPR tracking per user."
    },

    "TCR": {
        "name": "Team Contribution Rank",
        "dimension": "Relative Standing & Equity",
        "formula": "(Upload% + Created% + Published%) / 3 per user. Gini coefficient for publish distribution inequality.",
        "benchmark": "Gini ≤ 0.50",
        "current_avg": "Gini 0.74 (high inequality)",
        "status": "WARNING",
        "priority": "MONITOR",
        "datasets_required": ["cmdbu"],
        "description": "Composite ranking combining upload share, created share and published share into single normalised rank score. Gini coefficient measures inequality in published output distribution.",
        "interpretation": "Chandan tops at 14.18 composite score. Gini 0.74 means top 5 users account for 51.4% of all publishing.",
        "action": "Target Gini ≤ 0.50 by end of year through activating dormant publishers. Publish contribution charts per user on team dashboard."
    },

    "ICLS": {
        "name": "Idle Content Liability Score",
        "dimension": "Asset Management",
        "formula": "Created Duration (hrs) - Published Duration (hrs) (per user)",
        "benchmark": "< 10 hours idle per user",
        "current_avg": "Chandan holds most at 208.09h idle",
        "status": "CRITICAL",
        "priority": "IMMEDIATE",
        "datasets_required": ["cmdbu"],
        "description": "Total hours of AI-created content each user holds in unpublished state — their personal content liability. Translates abstract DPOR percentage into actual hours of locked value.",
        "interpretation": "Top 5 idle users: Chandan (208h) + Trivendra (111h) + vikas.s (102h) + Nitesh (88h) + Sandeep (84h) = 593h idle combined.",
        "action": "Activate 60-day backlog clearance sprint targeting top 5 idle-hour users. Even publishing 1% of Trivendra's 111h = 66 minutes of new distributed content."
    },

    "UCFCR": {
        "name": "User Content Funnel Completion Rate",
        "dimension": "End-to-End Delivery",
        "formula": "(Published Count / Uploaded Count) × 100 (per user)",
        "benchmark": "≥ 15% per user",
        "current_avg": "2.49% team avg",
        "status": "CRITICAL",
        "priority": "IMMEDIATE",
        "datasets_required": ["cmdbu"],
        "description": "Percentage of each user's raw uploads that result in at least one published piece. Most comprehensive single-number summary of user operational effectiveness.",
        "interpretation": "QA-Amit leads at 60%. Only 5 users exceed 5% UCFCR. 32 users have never completed a single full funnel.",
        "action": "Progressive targets: all users ≥1% UCFCR within 30 days, ≥5% within 90 days, ≥15% within 6 months."
    }
}

In [ ]:
client = OpenAI(
    api_key="API_KEY",
    base_url="https://api.groq.com/openai/v1"
)

def model(prompt, role, temperature, token):

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": role},
            {"role": "user", "content": prompt}
        ],
        max_tokens=token,
        temperature=temperature
    )

    return response.choices[0].message.content

In [ ]:
# context resolver

def context_resolver(query, recent_queries):

    if len(recent_queries) == 0:
        return query

    kpi_reference = "\n".join([
        f"- {abbr}: {details['name']} — {details['description']}"
        for abbr, details in kpi_framework.items()
    ])

    role = f"""
You are a query context resolver for a video analytics dashboard assistant. Your job is to determine whether the current user query depends on previous queries.
If it depends on previous queries, rewrite the query so that it becomes fully self-contained and clear.
If it does NOT depend on previous queries, return the original query unchanged.
The dashboard tracks video content performance across upload, creation and publishing stages.
Users may refer to KPIs by their short names or abbreviations. Here are the known KPIs: {kpi_reference}
Conversation history: {recent_queries}
Rules:
- If the user refers to a KPI by abbreviation (e.g. "DPOR", "ETR", "CPCR"), expand it to its full name in the rewritten query.
- If the user says "this", "that", "it", "same", "those" etc., resolve what they refer to using conversation history.
- If the query is completely independent of history, return it unchanged.
- Return ONLY the rewritten query as plain text. No explanations, no prefixes, no quotes.
"""
    prompt = f"""User query: {query}"""
    response = model(prompt, role, temperature = 0.4, token = 500)
    return response

In [ ]:
import ast
def choose_dataset(query):

  kpi_reference = "\n".join([
        f"- {abbr}: {details['name']} — {details['description']} - {details["datasets_required"]}"
        for abbr, details in kpi_framework.items()
    ])

  role = f"""
          You are selecting datasets required to answer a query. Query may require more than one datasets.
          Available datasets: {disc}, KPIs info: {kpi_reference}
          Rules:
            Return ONLY a Python list of dataset name strings.
            Example output: ['cmd', 'mc']
            No explanations, no extra text, just the list."""
  prompt = f""" User query: {query}"""

  response = model(prompt, role, temperature = 0.1, token = 80)
  dnames = ast.literal_eval(response)
  return dnames

In [ ]:
# precompute schema cache once at startup
schema_cache = {}
for name in datasets:
    df = datasets[name]
    schema_cache[name] = {
        "columns": list(df.columns),
        "dtypes": df.dtypes.astype(str).to_dict(),
        "row_count": len(df),
        "sample_rows": df.head(3).to_dict(orient="records")
    }

In [ ]:
def extract_schema(dataset_names):

    schema = {}
    for name in dataset_names:
        if name not in datasets:
            raise RuntimeError(f"Dataset '{name}' not found in datasets dictionary")
        schema[name] = schema_cache[name]

    return schema

In [ ]:
def interpret_query(query, schema):

    kpi_reference = "\n".join([
        f"- {abbr} | {details['name']} | Formula: {details['formula']} | Datasets: {details['datasets_required']}"
        for abbr, details in kpi_framework.items()
    ])

    role = f"""
You are an AI data analyst for a video analytics dashboard.
Your job is to convert a user query into a structured data analysis plan that can be executed by downstream tools.
Available dataset descriptions: {disc}
Available dataset schemas: {schema}
Known KPIs and their formulas: {kpi_reference}

Using the user query, extract the following fields:
* task: the main analytical objective of the query
* is_kpi_query: true if the query is asking about a specific KPI, false otherwise
* kpi_name: short abbreviation of the KPI (e.g. "DPOR", "ETR") or null if not a KPI query
* kpi_formula: the exact formula string to compute or null if not a KPI query
* metrics: numerical measures to compute or analyze
* operations: none as the provided datasets are already aggregated
* filters: conditions that restrict the data
* sorting: ordering of results if requested
* output: the expected analytical output (table, number, chart insight, etc.)
Example output format:
{{
  "task": "...",
  "is_kpi_query": true,
  "kpi_name": "DPOR",
  "kpi_formula": "(Created Count - Published Count) / Created Count × 100",
  "metrics": [...],
  "operations": [...],
  "filters": {{}},
  "sorting": {{}},
  "output": "..."
}}

Rules:
* Use only columns present in the schemas.
* Do not generate SQL queries or plotting code.
* Return ONLY valid JSON.
* Do NOT wrap the output in ``` or any code block.
* Do not include explanations or additional text."""

    prompt = f"""User query: {query}"""
    response = model(prompt, role, temperature = 0, token = 250)
    plan = json.loads(response)
    return plan

In [ ]:
def generate_sql(query, schema, plan):

    kpi_instruction = ""
    if plan.get("is_kpi_query") and plan.get("kpi_formula"):
        kpi_instruction = f"""
IMPORTANT: This is a KPI query for {plan.get("kpi_name")}. Use exactly this formula to compute it: {plan.get("kpi_formula")}
Do not approximate or simplify the formula.
"""

    role = f"""
You are an expert SQL generator for DuckDB.
Available datasets and schemas: {schema}
Analysis plan: {plan}, {kpi_instruction}

Rules:
1. Use only the provided dataset names and columns.
2. Dataset names are registered as DuckDB tables exactly as shown in the schema keys.
3. Always return both the dimension column and the metric column.
4. Use standard SQL fully compatible with DuckDB.
5. If a column name has spaces in it, always wrap it in double quotes.
6. Duration columns contain raw values in SECONDS stored as floats, despite their (hh:mm:ss) column name label.
7. Return ONLY the raw SQL query.
8. Do NOT wrap output in ``` or any code block.
9. Do not include explanations or comments. """

    prompt = f"""{query}"""
    response = model(prompt, role, temperature=0, token=300)
    return response


In [ ]:
import duckdb

def run_sql(sql_query, dataset_names):

    # remove markdown formatting if present
    sql_query = sql_query.replace("```sql", "").replace("```", "").strip()

    # create DuckDB connection
    con = duckdb.connect()

    # register pandas dataframes as SQL tables
    for name in dataset_names:
        con.register(name, datasets[name])

    # execute SQL and return result as dataframe
    result = con.execute(sql_query).df()

    return result
    con.close()

In [ ]:
def fix_sql(failed_sql, error_message, query, schema, plan = None):

    kpi_instruction = ""
    if plan.get("is_kpi_query") and plan.get("kpi_formula"):
        kpi_instruction = f"""
IMPORTANT: This is a KPI query for {plan.get("kpi_name")}.
Use exactly this formula to compute it: {plan.get("kpi_formula")}
Do not approximate or simplify the formula.
"""
    role = f"""
You are an expert SQL debugger for DuckDB.
A SQL query was generated but failed to execute. Your job is to fix it.

Original user query: {query}
Available dataset schemas: {schema}, {kpi_instruction}

Rules:
1. Use only the provided dataset names and columns.
2. If a column name has spaces in it, always wrap it in double quotes.
3. Duration columns contain raw values in SECONDS stored as floats.
4. Use standard SQL fully compatible with DuckDB.
5. Return ONLY the fixed raw SQL query.
6. Do NOT wrap output in ``` or any code block.
7. Do not include explanations or comments.
"""

    prompt = f"""
The following SQL query failed: {failed_sql}
DuckDB error: {error_message}
Fix the SQL query so it executes correctly.
"""

    response = model(prompt, role, temperature=0, token=500)

    # Strip markdown fences if present
    response = re.sub(r"```(?:sql)?", "", response).replace("```", "").strip()

    if not response:
        raise RuntimeError("fix_sql returned an empty response")

    return response

In [ ]:
def update_result_memory(result_memory, sql_query, result, plan = None):

    result_memory["last_sql"] = sql_query
    result_memory["last_result"] = result
    result_memory["last_plot"] = None

    if result is not None and not result.empty:
        result_memory["columns"] = list(result.columns)
        result_memory["row_count"] = len(result)
        result_memory["summary"] = result.describe(include="all").to_dict()
    else:
        result_memory["columns"] = None
        result_memory["row_count"] = 0
        result_memory["summary"] = None

    if plan and plan.get("is_kpi_query"):
        result_memory["last_kpi"] = {
            "kpi_name": plan.get("kpi_name"),
            "kpi_formula": plan.get("kpi_formula")
        }
    else:
        result_memory["last_kpi"] = None

    return result_memory

In [ ]:
def get_result_context(result_memory):

    if result_memory["last_result"] is None:
        return None

    context = {
        "columns": result_memory["columns"],
        "row_count": result_memory["row_count"],
        "last_sql": result_memory["last_sql"],
        "summary": result_memory["summary"],
        "last_kpi": result_memory["last_kpi"],
        "preview": result_memory["last_result"].head(5).to_dict(orient="records")
    }

    return context

In [ ]:
def plotting_agent(query, result, result_context):
  role = f"""

Your task is to determine whether a plot is required to answer or explain the user query
based on the dataframe returned from a database query.

If plotting helps explain the result OR the user explicitly asks for a chart,
recommend an appropriate plot.

Return ONLY valid JSON.

The JSON must follow this schema:

{{
"requires_plot": true or false,
"plot_type": "bar | line | scatter | histogram | none",
"x_column": "column name or null",
"y_column": "column name or null",
"series_column": "column name or null",
"x_label": "label",
"y_label": "label",
"title": "chart title"
}}
Previous Result Context:
{result_context}

Rules:
- Use only columns present in the dataframe.
- If the query references the previous result (e.g. "plot this", "visualize it"), use the Previous Result Context.
- If the query does not refer to previous results, ignore the Previous Result Context.
- If visualization is unnecessary or only one row, set requires_plot=false and plot_type=none.
- Do NOT wrap the output in ``` or any code block.
- DO NOT include explanations.
- DO NOT include markdown.
- DO NOT include text outside JSON
  """
  prompt = f"""
User Query:
{query}
Dataset:
{result}
"""
  response = model(prompt, role, temperature = 0.1, token = 400)
  plan = json.loads(response)
  return plan

In [ ]:
from matplotlib.ticker import FuncFormatter

def plotting_tool(df, plot_spec):

  if not plot_spec["requires_plot"]:
    return None

  plot_type = plot_spec.get("plot_type")
  x = plot_spec.get("x_column")
  y = plot_spec.get("y_column")
  series = plot_spec.get("series_column")
  xlabel = plot_spec.get("x_label", "")
  ylabel = plot_spec.get("y_label", "")
  title = plot_spec.get("title", "")

  fig, ax = plt.subplots(figsize=(8,5))
  plt.xticks(rotation=90)

  def seconds_formatter(value, pos):
      value = int(value)
      hours = value // 3600
      minutes = (value % 3600) // 60
      seconds = value % 60
      return f"{hours:02}:{minutes:02}:{seconds:02}"

  # LINE
  if plot_type == "line":
      ax.plot(df[x], df[y], marker='o')

  # BAR
  elif plot_type == "bar":
      ax.bar(df[x], df[y])

  # SCATTER
  elif plot_type == "scatter":
      ax.scatter(df[x], df[y])

  # HISTOGRAM
  elif plot_type == "histogram":
      ax.hist(df[x], bins=20)

  else:
      return None

  ax.set_xlabel(xlabel)
  ax.set_ylabel(ylabel)
  ax.set_title(title)

  # Format duration axis
  if y and pd.api.types.is_numeric_dtype(df[y]):
      if "time" in y.lower() or "duration" in y.lower():
          ax.yaxis.set_major_formatter(FuncFormatter(seconds_formatter))

  if plot_type == "histogram" and x:
      if "time" in x.lower() or "duration" in x.lower():
          ax.xaxis.set_major_formatter(FuncFormatter(seconds_formatter))

  fig.tight_layout()
  return fig

In [ ]:
def update_plot_memory(result_memory, plot_spec):
    result_memory["last_plot"] = plot_spec
    return result_memory

In [ ]:
def convert_seconds_to_hms(df):

    df = df.copy()

    def format_time(sec):
        if pd.isna(sec):
            return "00:00:00" # Handle NaN values
        sec = int(sec)
        hours = sec // 3600
        minutes = (sec % 3600) // 60
        seconds = sec % 60
        return f"{hours:02}:{minutes:02}:{seconds:02}"

    for col in df.columns:
        if "time" in col.lower() or "duration" in col.lower():
            df[col] = df[col].apply(format_time)

    return df

In [ ]:
def explain_result(query, result, result_context, plot_plan):

    kpi_context = ""
    if result_context and result_context.get("last_kpi"):
        kpi_name = result_context["last_kpi"]["kpi_name"]
        kpi_formula = result_context["last_kpi"]["kpi_formula"]

        if kpi_name and kpi_name in kpi_framework:
            kpi_info = kpi_framework[kpi_name]
            kpi_context = f"""
This query is about the KPI: {kpi_info['name']} ({kpi_name})
Dimension: {kpi_info['dimension']}
Formula: {kpi_formula}
Benchmark / Target: {kpi_info['benchmark']}
Current Team Average: {kpi_info['current_avg']}
Status: {kpi_info['status']} | Priority: {kpi_info['priority']}
Interpretation guidance: {kpi_info['interpretation']}
Recommended actions: {kpi_info['action']}

When explaining:
- Compare results against the benchmark: {kpi_info['benchmark']}
- Highlight users or channels that are above and below benchmark
- Reference the status ({kpi_info['status']}) and what it means for the team
- Suggest actionable next steps based on the recommended actions above"""

    plot_info = plot_plan if plot_plan.get("requires_plot") else None
    role = f"""
You are an AI data analyst for a video analytics platform where users use AI tools to clip and edit videos.
Your job is to explain query results clearly and concisely to dashboard users.
Rules:
- Do not start your answer with "Based on the provided dataset" or any similar phrase.
- Do not ask the user questions or cross-question them — just explain the result.
- If the user query references previous results (e.g. "this", "that", "it"), use the Previous Result Context to understand what is being referred to.
- If the query does not refer to previous results, ignore the Previous Result Context.
- Use the plot info to explain what the chart shows if a plot was generated.
- End your response with 3-4 relevant follow-up questions the user might want to ask, under a heading "Follow-up Questions:". Questions must be relevant to the available datasets.
- Keep the explanation concise and insight-focused.

{kpi_context}
Previous Result Context: {result_context}
Dataset descriptions for follow-up question context: {disc}"""

    prompt = f"""
User question: {query}
Query result: {result.to_dict(orient="records")}
Plot info: {plot_info}
"""

    response = model(prompt, role, temperature=0.4, token=800)
    return response

In [ ]:
def update_memory(memory, query, resolved_query, plan = None, max_history=6):

    memory["last_user_query"] = query
    memory["last_resolved_query"] = resolved_query

    if plan and plan.get("is_kpi_query"):
        memory["last_kpi"] = plan.get("kpi_name")
    else:
        memory["last_kpi"] = None
    # only store valid resolved queries
    if resolved_query:
        memory["recent_queries"].append(resolved_query)

    # keep only last N queries
    memory["recent_queries"] = memory["recent_queries"][-max_history:]

    return memory

In [ ]:
result_memory = {"last_sql": None,"last_result": None, "columns": None, "row_count": None, "last_plot": None, "summary": None, "last_kpi": None}
memory = {"recent_queries": [], "last_user_query": None, "last_resolved_query": None, "last_kpi": None}

def chatbot_ui(query):
    global result_memory, memory
    try:
        resolved_query = context_resolver(query, memory["recent_queries"])
        listd = choose_dataset(resolved_query)
        schema = extract_schema(listd)
        plan = interpret_query(resolved_query, schema)
        sql_query = generate_sql(resolved_query, schema, plan)
        try:
                result = run_sql(sql_query, listd)
        except Exception as sql_error:
                fixed_sql = fix_sql(sql_query, str(sql_error), resolved_query, schema, plan)
                result = run_sql(fixed_sql, listd)
        result_memory = update_result_memory(result_memory, sql_query, result, plan)
        result_context = get_result_context(result_memory)
        plot_spec = plotting_agent(resolved_query, result, result_context)
        plot_fig = plotting_tool(result, plot_spec)
        result_memory = update_plot_memory(result_memory, plot_spec)
        result = convert_seconds_to_hms(result)
        explanation = explain_result(resolved_query, result, result_context, plot_spec)
        memory = update_memory(memory, query, resolved_query, plan)
        return explanation, plot_fig
    except Exception as e:
        raise RuntimeError(f"chatbot_ui failed at query '{query}': {str(e)}")

In [ ]:
!pip install fastapi uvicorn pyngrok nest_asyncio

In [ ]:
from fastapi import FastAPI, HTTPException
import traceback
app = FastAPI()

import base64
import io
import threading
lock = threading.Lock()

@app.post("/reset")
def reset():
    with lock:
        global memory, result_memory
        memory = {"recent_queries": [], "last_user_query": None, "last_resolved_query": None, "last_kpi": None }
        result_memory = {"last_sql": None, "last_result": None, "columns": None, "row_count": None, "last_plot": None, "summary": None, "last_kpi": None}
        return {"status": "reset successful"}

@app.post("/run_query")
def run_query(data: dict):
  with lock:
    try:
        query = data["query"]
        explanation, fig = chatbot_ui(query)
    except Exception as e:
       traceback.print_exc()
       raise HTTPException(status_code=500, detail=str(e))

    plot_base64 = None
    if fig is not None:
        buf = io.BytesIO()
        fig.savefig(buf, format="png")
        plt.close(fig)
        buf.seek(0)
        plot_base64 = base64.b64encode(buf.read()).decode("utf-8")

    return {
        "explanation": explanation,
        "plot": plot_base64
    }

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import threading

nest_asyncio.apply()

# Add your ngrok auth token
ngrok.set_auth_token("NGROK_API_KEY")
ngrok.kill()
public_url = ngrok.connect(8000)

print("Public URL:", public_url.public_url)

def run_uvicorn():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = uvicorn.Server(config)
    server.run()

uvicorn_thread = threading.Thread(target=run_uvicorn)
uvicorn_thread.start()


Public URL: https://chignoned-unsurprised-alex.ngrok-free.dev
